In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Data Engineering\Projects\fleetpulse_project")

if str(PROJECT_ROOT) is not sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [15]:
from scripts.utils.spark_session import create_spark_session
from scripts.utils.config import (
    BRONZE_BASE_PATH,
    SILVER_BASE_PATH
)
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [16]:
table = "equipment"
bronze_table_path = f"{BRONZE_BASE_PATH}/{table}"
spark = create_spark_session("LoadingEquipementTableToSilver")

In [17]:
df = spark.read.parquet(bronze_table_path)
df.show(truncate=False)

+------------+----------------+-------------+-----------+----------------+-----------------+------------+-------+----------+----------------+----------------+----------------+------------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|equipment_id|equipment_number|serial_number|category_id|owning_branch_id|current_branch_id|manufacturer|model  |model_year|acquisition_date|acquisition_cost|equipment_status|telematics_enabled|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+------------+----------------+-------------+-----------+----------------+-----------------+------------+-------+----------+----------------+----------------+----------------+------------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------

In [18]:
df_clean = (
            df
            .withColumn("equipment_number",F.upper(F.trim(F.col("equipment_number"))))
            .withColumn("serial_number", F.upper(F.trim(F.col("serial_number"))))
            .withColumn("manufacturer", F.trim(F.col("manufacturer")))
            .withColumn("model", F.trim(F.col("model")))
            .withColumn("equipment_status",F.trim(F.upper(F.coalesce(F.col("equipment_status"),F.lit("UNKNOWN")))))
            .withColumn("telematics_enabled",F.coalesce(F.col("telematics_enabled"),F.lit(False)))
            .withColumn("source_system",F.trim(F.upper(F.col("source_system"))))
            .withColumn("is_deleted",F.coalesce("is_deleted",F.lit(False)))
            .withColumn("source_table",F.trim(F.col("source_table")))
            .withColumn("bronze_load_type", F.trim(F.upper("bronze_load_type")))
)

df_clean.show(truncate=False)

+------------+----------------+-------------+-----------+----------------+-----------------+------------+-------+----------+----------------+----------------+----------------+------------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|equipment_id|equipment_number|serial_number|category_id|owning_branch_id|current_branch_id|manufacturer|model  |model_year|acquisition_date|acquisition_cost|equipment_status|telematics_enabled|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+------------+----------------+-------------+-----------+----------------+-----------------+------------+-------+----------+----------------+----------------+----------------+------------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------

In [19]:
window_spec = (
                Window
                .partitionBy("equipment_id")
                .orderBy(
                    F.col("updated_at").desc(),
                    F.col("ingestion_timestamp").desc()
                )
            )
df_dedup = (
        df_clean
        .withColumn("record_num",F.row_number().over(window_spec))
        .filter(F.col("record_num") == 1)
        .drop(F.col("record_num"))
)


In [20]:
df_hash = (
            df_clean
            .withColumn("record_hash", 
                        F.sha2(
                            F.concat_ws(
                                "|",
                                F.col("category_id"),
                                F.col("owning_branch_id"),
                                F.col("current_branch_id"),
                                F.col("model_year"),
                                F.col("acquisition_date"),
                                F.col("acquisition_cost"),
                                F.col("telematics_enabled"),
                                F.col("is_deleted")
                            ),256))
)

df_hash.show(truncate=False)

+------------+----------------+-------------+-----------+----------------+-----------------+------------+-------+----------+----------------+----------------+----------------+------------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+----------------------------------------------------------------+
|equipment_id|equipment_number|serial_number|category_id|owning_branch_id|current_branch_id|manufacturer|model  |model_year|acquisition_date|acquisition_cost|equipment_status|telematics_enabled|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|record_hash                                                     |
+------------+----------------+-------------+-----------+----------------+-----------------+------------+-------+----------+----------------+----------------+----------------+-------

In [21]:
silver_df = (
                df_hash
                .withColumn("silver_created_timestamp", F.current_timestamp())
                .withColumn("silver_updated_timestamp", F.current_timestamp())
                .withColumn("silver_quality_status", F.lit("VALID"))
                )

In [22]:
table = "rental_contracts"
bronze_rental_table_path = f"{BRONZE_BASE_PATH}/{table}"
rental_contracts_df = spark.read.parquet(bronze_rental_table_path)
rental_contracts_df.show(truncate=False)


+-----------+---------------+-----------+---------+---------------------+-----------------+--------------------+------------------+---------------+-------------+-------------+---------------+---------------+----------+------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+----------------+----------------+
|contract_id|contract_number|customer_id|branch_id|contract_created_date|rental_start_date|expected_return_date|actual_return_date|contract_status|sales_channel|payment_terms|subtotal_amount|discount_amount|tax_amount|total_amount|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table    |bronze_load_type|
+-----------+---------------+-----------+---------+---------------------+-----------------+--------------------+------------------+---------------+-------------+-------------+---------------+---------------+----------+------------

In [29]:
silver_base_path = SILVER_BASE_PATH

from scripts.config.silver_table_config import SILVER_TABLE_CONFIG

def apply_referential_checks(df,spark,referential_checks,silver_base_path):
    result_df = df
    print(referential_checks)
    for check in referential_checks:
        parent_table = check['parent_table']
        child_key = check['child_key']
        parent_key = check['parent_key']
        parent_df = spark.read.parquet(f"{silver_base_path}/{parent_table}")
        parent_key_df = ( 
                        parent_df
                        .select(F.col(parent_key).alias("__parent_key"))
                        .distinct() 
                         )
        result_df = ( 
                    result_df
                    .join(
                        parent_key_df, 
                        result_df[child_key] == parent_key_df["__parent_key"],
                        how='inner')
                    .drop("__parent_key") )
    return result_df

df = apply_referential_checks(spark=spark,df=rental_contracts_df, referential_checks=SILVER_TABLE_CONFIG['rental_contracts']['referential_checks'],silver_base_path=silver_base_path)
df.show(truncate=False)

[{'parent_table': 'customers', 'child_key': 'customer_id', 'parent_key': 'customer_id'}, {'parent_table': 'branches', 'child_key': 'branch_id', 'parent_key': 'branch_id'}]
+-----------+---------------+-----------+---------+---------------------+-----------------+--------------------+------------------+---------------+-------------+-------------+---------------+---------------+----------+------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+----------------+----------------+
|contract_id|contract_number|customer_id|branch_id|contract_created_date|rental_start_date|expected_return_date|actual_return_date|contract_status|sales_channel|payment_terms|subtotal_amount|discount_amount|tax_amount|total_amount|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table    |bronze_load_type|
+-----------+---------------+-----------+---------+-------